# Exploratory Data Analysis using Python - A Case Study

Let's load the CSV files using the Pandas libray. We'll use the name `survey_raw_df` for the data frame, to indicate that this is unprocessed data that which we might clean, filter and modify to prepare a data frame that's ready for analysis.

In [1]:
import pandas as pd

In [2]:
survey_raw_df = pd.read_csv('survey_results_public.csv')

In [3]:
survey_raw_df.head()

,Respondent,MainBranch,Hobbyist,Age,Age1stCode,CompFreq,CompTotal,ConvertedComp,Country,CurrencyDesc,...,SurveyEase,SurveyLength,Trans,UndergradMajor,WebframeDesireNextYear,WebframeWorkedWith,WelcomeChange,WorkWeekHrs,YearsCode,YearsCodePro
0,1,I am a developer by profession,Yes,NaN,13,Monthly,NaN,NaN,Germany,European Euro,...,Neither easy nor difficult,Appropriate in length,No,"Computer science, computer engineering, or sof...",ASP.NET Core,ASP.NET;ASP.NET Core,Just as welcome now as I felt last year,50.0,36,27
1,2,I am a developer by profession,No,NaN,19,NaN,NaN,NaN,United Kingdom,Pound sterling,...,NaN,NaN,NaN,"Computer science, computer engineering, or sof...",NaN,NaN,Somewhat more welcome now than last year,NaN,7,4
2,3,I code primarily as a hobby,Yes,NaN,15,NaN,NaN,NaN,Russian Federation,NaN,...,Neither easy nor difficult,Appropriate in length,NaN,NaN,NaN,NaN,Somewhat more welcome now than last year,NaN,4,NaN
3,4,I am a developer by profession,Yes,25.0,18,NaN,NaN,NaN,Albania,Albanian lek,...,NaN,NaN,No,"Computer science, computer engineering, or sof...",NaN,NaN,Somewhat less welcome now than last year,40.0,7,4
4,5,"I used to be a developer by profession, but no...",Yes,31.0,16,NaN,NaN,NaN,United States,NaN,...,Easy,Too short,No,"Computer science, computer engineering, or sof...",Django;Ruby on Rails,Ruby on Rails,Just as welcome now as I felt last year,NaN,15,8


The dataset contains over 64,000 responses to 60 questions (although many questions are optional). The responses have been anonymized and there's no personally identifiable information available to us - although each respondent has been assigned a randomized respondent ID.

Let's view the list of columns in the data frame.

In [4]:
survey_raw_df.columns

Index(['Respondent', 'MainBranch', 'Hobbyist', 'Age', 'Age1stCode', 'CompFreq',
       'CompTotal', 'ConvertedComp', 'Country', 'CurrencyDesc',
       'CurrencySymbol', 'DatabaseDesireNextYear', 'DatabaseWorkedWith',
       'DevType', 'EdLevel', 'Employment', 'Ethnicity', 'Gender', 'JobFactors',
       'JobSat', 'JobSeek', 'LanguageDesireNextYear', 'LanguageWorkedWith',
       'MiscTechDesireNextYear', 'MiscTechWorkedWith',
       'NEWCollabToolsDesireNextYear', 'NEWCollabToolsWorkedWith', 'NEWDevOps',
       'NEWDevOpsImpt', 'NEWEdImpt', 'NEWJobHunt', 'NEWJobHuntResearch',
       'NEWLearn', 'NEWOffTopic', 'NEWOnboardGood', 'NEWOtherComms',
       'NEWOvertime', 'NEWPurchaseResearch', 'NEWPurpleLink', 'NEWSOSites',
       'NEWStuck', 'OpSys', 'OrgSize', 'PlatformDesireNextYear',
       'PlatformWorkedWith', 'PurchaseWhat', 'Sexuality', 'SOAccount',
       'SOComm', 'SOPartFreq', 'SOVisitFreq', 'SurveyEase', 'SurveyLength',
       'Trans', 'UndergradMajor', 'WebframeDesireNextYear',
  

It appears that short codes for questions are used as column names.

We can refer to the schema file to see the full text of each question. The schema file contains only two columns: `Column` and `QuestionText`, so we can load it as Pandas Series with `Column` as the index and the `QuestionText` as the value.

In [5]:
schema_fname = 'survey_results_schema.csv'

In [9]:
schema_raw = pd.read_csv(schema_fname, index_col='Column').QuestionText

schema_raw.Age

'What is your age (in years)? If you prefer not to answer, you may leave this question blank.'

In [ ]:
# Since we have one column we do not need a data frame, instead we can retrieve just the texts.
schema_raw = pd.read_csv(schema_fname, index_col='Column').QuestionText

In [10]:
schema_raw

Column
Respondent            Randomized respondent ID number (not in order ...
MainBranch            Which of the following options best describes ...
Hobbyist                                        Do you code as a hobby?
Age                   What is your age (in years)? If you prefer not...
Age1stCode            At what age did you write your first line of c...
                                            ...                        
WebframeWorkedWith    Which web frameworks have you done extensive d...
WelcomeChange         Compared to last year, how welcome do you feel...
WorkWeekHrs           On average, how many hours per week do you wor...
YearsCode             Including any education, how many years have y...
YearsCodePro          NOT including education, how many years have y...
Name: QuestionText, Length: 61, dtype: object

In [11]:
type(schema_raw)

pandas.core.series.Series

We can now use `schema_raw` to retrieve the full question text for any column in `survey_raw_df`.

In [12]:
schema_raw['MainBranch']

'Which of the following options best describes you today? Here, by "developer" we mean "someone who writes code."'

In [13]:
survey_raw_df['MainBranch'].unique()

array(['I am a developer by profession', 'I code primarily as a hobby',
       'I used to be a developer by profession, but no longer am',
       'I am not primarily a developer, but I write code sometimes as part of my work',
       'I am a student who is learning to code', nan], dtype=object)

In [14]:
schema_raw['Hobbyist']

'Do you code as a hobby?'

In [15]:
survey_raw_df['Hobbyist'].unique()

array(['Yes', 'No', nan], dtype=object)

In [16]:
schema_raw['YearsCodePro']

'NOT including education, how many years have you coded professionally (as a part of your work)?'

We've now loaded the dataset, and we're ready to move on to the next step of preprocessing & cleaning the data for our analysis.

## Data Preparation & Cleaning

While the survey responses contain a wealth of information, we'll limit our analysis to the following areas:

- Demographics of the survey respondents & the global programming community
- Distribution of programming skills, experience and preferences
- Employment-related information, preferences & opinions

Let's select a subset of columns with the relevant data for our analysis.

In [17]:
selected_columns = [
    # Demographics
    'Country',
    'Age',
    'Gender',
    'EdLevel',
    'UndergradMajor',
    # Programming experience
    'Hobbyist',
    'Age1stCode',
    'YearsCode',
    'YearsCodePro',
    'LanguageWorkedWith',
    'LanguageDesireNextYear',
    'NEWStuck',
    'NEWLearn',
    # Employment
    'Employment',
    'DevType',
    'WorkWeekHrs',
    'JobSat',
    'JobFactors',
    'NEWOvertime',
    'NEWEdImpt'
]

In [18]:
len(selected_columns)

20

Let's extract a copy of the data from these columns into a new data frame `survey_df`, which we can continue to modify further without affecting the original data frame.

In [19]:
survey_df = survey_raw_df[selected_columns].copy()

In [20]:
schema = schema_raw[selected_columns]

In [21]:
schema.shape

(20,)

Let's view some basic information about the data frame.

In [22]:
survey_df.shape

(64461, 20)

In [23]:
schema['EdLevel']

'Which of the following best describes the highest level of formal education that you’ve completed?'

In [24]:
survey_df['EdLevel'].value_counts()

EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          26542
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       13112
Some college/university study without earning a degree                                 7239
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     4771
Associate degree (A.A., A.S., etc.)                                                    1843
Other doctoral degree (Ph.D., Ed.D., etc.)                                             1690
Primary/elementary school                                                               941
Professional degree (JD, MD, etc.)                                                      800
I never completed any formal education                                                  493
Name: count, dtype: int64

So many people have a Bachelor's degree, meaning a huge number of coders hear leant from a University or College.

In [26]:
survey_df.head()

,Country,Age,Gender,EdLevel,UndergradMajor,Hobbyist,Age1stCode,YearsCode,YearsCodePro,LanguageWorkedWith,LanguageDesireNextYear,NEWStuck,NEWLearn,Employment,DevType,WorkWeekHrs,JobSat,JobFactors,NEWOvertime,NEWEdImpt
0,Germany,NaN,Man,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Computer science, computer engineering, or sof...",Yes,13,36,27,C#;HTML/CSS;JavaScript,C#;HTML/CSS;JavaScript,Visit Stack Overflow;Go for a walk or other ph...,Once a year,"Independent contractor, freelancer, or self-em...","Developer, desktop or enterprise applications;...",50.0,Slightly satisfied,"Languages, frameworks, and other technologies ...",Often: 1-2 days per week or more,Fairly important
1,United Kingdom,NaN,NaN,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Computer science, computer engineering, or sof...",No,19,7,4,JavaScript;Swift,Python;Swift,Visit Stack Overflow;Go for a walk or other ph...,Once a year,Employed full-time,"Developer, full-stack;Developer, mobile",NaN,Very dissatisfied,NaN,NaN,Fairly important
2,Russian Federation,NaN,NaN,NaN,NaN,Yes,15,4,NaN,Objective-C;Python;Swift,Objective-C;Python;Swift,NaN,Once a decade,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Albania,25.0,Man,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Computer science, computer engineering, or sof...",Yes,18,7,4,NaN,NaN,NaN,Once a year,NaN,NaN,40.0,Slightly dissatisfied,Flex time or a flexible schedule;Office enviro...,Occasionally: 1-2 days per quarter but less th...,Not at all important/not necessary
4,United States,31.0,Man,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Computer science, computer engineering, or sof...",Yes,16,15,8,HTML/CSS;Ruby;SQL,Java;Ruby;Scala,Call a coworker or friend;Visit Stack Overflow...,Once a year,Employed full-time,NaN,NaN,NaN,NaN,NaN,Very important


In [28]:
survey_df.isnull().sum()

Country                     389
Age                       19015
Gender                    13904
EdLevel                    7030
UndergradMajor            13466
Hobbyist                     45
Age1stCode                 6561
YearsCode                  6777
YearsCodePro              18112
LanguageWorkedWith         7083
LanguageDesireNextYear    10348
NEWStuck                   9478
NEWLearn                   8305
Employment                  607
DevType                   15091
WorkWeekHrs               23310
JobSat                    19267
JobFactors                15112
NEWOvertime               21230
NEWEdImpt                 15996
dtype: int64

We have so many missing values from this data set.

In [27]:
survey_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64461 entries, 0 to 64460
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Country                 64072 non-null  object 
 1   Age                     45446 non-null  float64
 2   Gender                  50557 non-null  object 
 3   EdLevel                 57431 non-null  object 
 4   UndergradMajor          50995 non-null  object 
 5   Hobbyist                64416 non-null  object 
 6   Age1stCode              57900 non-null  object 
 7   YearsCode               57684 non-null  object 
 8   YearsCodePro            46349 non-null  object 
 9   LanguageWorkedWith      57378 non-null  object 
 10  LanguageDesireNextYear  54113 non-null  object 
 11  NEWStuck                54983 non-null  object 
 12  NEWLearn                56156 non-null  object 
 13  Employment              63854 non-null  object 
 14  DevType                 49370 non-null

Most columns have the data type object, either because they contain values of different types, or they contain empty values, which are represented using `NaN`. It appears that every column contains some empty values, since Non-Null count for every column is lower than the total number of rows (66461). We'll need to deal with empty values and manually adjust the data type for each column on a case-by-case basis.

Only two of the columns were detected as numeric columns (Age and WorkWeekHrs), even though there are a few other columns which have mostly numeric values. To make our analysis easier, let's convert some other columns into numeric data types, while ignoring any non-numeric value (they will get converted to NaNs)

In [29]:
schema.Age1stCode

'At what age did you write your first line of code or program? (e.g., webpage, Hello World, Scratch project)'

In [30]:
survey_df.Age1stCode.unique()

array(['13', '19', '15', '18', '16', '14', '12', '20', '42', '8', '25',
       '22', '30', '17', '21', '10', '46', '9', '7', '11', '6', nan, '31',
       '29', '5', 'Younger than 5 years', '28', '38', '23', '27', '41',
       '24', '53', '26', '35', '32', '40', '33', '36', '54', '48', '56',
       '45', '44', '34', 'Older than 85', '39', '51', '68', '50', '37',
       '47', '43', '52', '85', '64', '55', '58', '49', '76', '72', '73',
       '83', '63'], dtype=object)

In [ ]:
# errors='coerce' tells pandas to convert invalid values to NaN (missing) instead of crashing.

# Where it's used:

# Most commonly with:

#     pd.to_numeric() — converting to numbers

#     pd.to_datetime() — converting to dates

#     astype() — type conversion

In [31]:
survey_df.YearsCode.unique()

array(['36', '7', '4', '15', '6', '17', '8', '10', '35', '5', '37', '19',
       '9', '22', '30', '23', '20', '2', 'Less than 1 year', '3', '13',
       '25', '16', '43', '11', '38', '33', nan, '24', '21', '12', '40',
       '27', '50', '46', '14', '18', '28', '32', '44', '26', '42', '31',
       '34', '29', '1', '39', '41', '45', 'More than 50 years', '47',
       '49', '48'], dtype=object)

In [32]:
survey_df['Age1stCode'] = pd.to_numeric(survey_df.Age1stCode, errors='coerce')
survey_df['YearsCode'] = pd.to_numeric(survey_df.YearsCode, errors='coerce')
survey_df['YearsCodePro'] = pd.to_numeric(survey_df.YearsCodePro, errors='coerce')

Let's now view some basic statistics about the numeric columns.

In [33]:
survey_df.describe()

,Age,Age1stCode,YearsCode,YearsCodePro,WorkWeekHrs
count,45446.000000,57473.000000,56784.000000,44133.000000,41151.000000
mean,30.834111,15.476572,12.782051,8.869667,40.782174
std,9.585392,5.114081,9.490657,7.759961,17.816383
min,1.000000,5.000000,1.000000,1.000000,1.000000
25%,24.000000,12.000000,6.000000,3.000000,40.000000
50%,29.000000,15.000000,10.000000,6.000000,40.000000
75%,35.000000,18.000000,17.000000,12.000000,44.000000
max,279.000000,85.000000,50.000000,50.000000,475.000000


There seems to be a problem with the age column, as the minimum values is 1 and max value is 279. This is a common issue with surveys: responses may contain invalid values due to accidental or intentional errors while responding. A simple fix would be ignore the row where the value in the age column is higher than 100 years or lower than 10 years as invalid survey responses. This can be done using the `.drop` method.

In [34]:
survey_df.drop(survey_df[survey_df.Age < 10].index, inplace=True)
survey_df.drop(survey_df[survey_df.Age > 80].index, inplace=True)

In [35]:
survey_df.describe()

,Age,Age1stCode,YearsCode,YearsCodePro,WorkWeekHrs
count,45408.000000,57444.000000,56761.000000,44118.000000,41133.000000
mean,30.796512,15.475472,12.782897,8.869237,40.777532
std,9.385754,5.113278,9.489714,7.757985,17.802026
min,10.000000,5.000000,1.000000,1.000000,1.000000
25%,24.000000,12.000000,6.000000,3.000000,40.000000
50%,29.000000,15.000000,10.000000,6.000000,40.000000
75%,35.000000,18.000000,17.000000,12.000000,44.000000
max,80.000000,85.000000,50.000000,50.000000,475.000000


The same hold true for `WorkWeekHrs`. Let's ignore entries where the value for the column is higher than 140 hours. (~20 hours per day)

In [36]:
survey_df.drop(survey_df[survey_df.WorkWeekHrs > 140].index, inplace=True)

The gender column also allow picking multiple options, but to simplify our analysis, we'll remove values containing more than option.

In [ ]:
schema.Gender

In [ ]:
survey_df['Gender'].value_counts()

In [ ]:
import numpy as np

In [ ]:
survey_df.where(~(survey_df.Gender.str.contains(';', na=False)), np.nan, inplace=True)

# This line replaces rows with multiple gender selections (containing a semicolon) with NaN.

# 1. survey_df.Gender.str.contains(';', na=False)
#     Looks at the Gender column
#     Checks if each value contains a semicolon ;
#     Returns True if found, False if not
#     na=False treats missing values as False (not containing ;)

# 2. ~(survey_df.Gender.str.contains(';', na=False))
#     The ~ (tilde) means NOT (negation)
#     Flips True to False and False to True

# 3. survey_df.where(mask, np.nan, inplace=True)
#     .where() keeps values where mask is True
#     Replaces with np.nan where mask is False
#     inplace=True modifies the original DataFrame

In [ ]:
survey_df['Gender'].value_counts()

We've now cleaned up and prepared the dataset for analysis. Let's take a look at sample of rows from the data frame.

In [ ]:
schema.LanguageWorkedWith

In [ ]:
survey_df.sample(6)

## Exploratory Analysis and Visualization
Before we can ask interesting questions about the survey responses, it would help to understand what the demographics i.e. country, age, gender, education level, employment level etc. of the respondent look like. It's important to explore these variables in order to understand how representative the survey is of the worldwide programming community, as a survey of this scale generally tends to have some selection bias

Let's begin by importing `matplotlib.pyplot` and `seaborn`.

In [ ]:
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

sns.set_style('darkgrid')
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (12,5)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

# Country
Let's look at the number of countries from which there are responses in the survey, and plot the 10 countries with the highest number of responses.

In [ ]:
schema.Country

In [ ]:
survey_df.Country.value_counts()

In [ ]:
survey_df.Country.nunique() # nunique() => number of unique values

We can identify the countries with the highest number of respondents using the `value_counts` method.

In [ ]:
top_countries = survey_df.Country.value_counts().head(15)
top_countries

In [ ]:
type(top_countries)

We can visualize this information using a bar chart.

In [ ]:
plt.xticks(rotation=75)
plt.title(schema.Country)
sns.barplot(x=top_countries.index, y=top_countries);

It appears that a disproportionately high number of respondents are from USA & India - which one might expect since these countries have the highest populations (apart from China), and since the survey is in English, which is the common language used by professionals in US, India & UK. We can already see that the survey may not be representative of the entire programming community - especially from non-English speaking countries.

# Age
The distributin of the age of respondents is another important factor to look at, and we can use a histogram to visualize it.

In [ ]:
schema.Age

In [ ]:
survey_df.Age[survey_df.Age.notnull()].agg(['mean', 'max', 'min', 'std', 'count'])

In [ ]:
plt.figure(figsize=(12,6))
plt.title(schema.Age)
plt.xlabel('Age')
plt.ylabel('Number of respondents')

plt.hist(survey_df.Age, bins=np.arange(10,80,5), color='purple');

It appears that a large percentage of respondents are in the age range of 20-45, which is somewhat representative of the programming community in general, as a lot of young people have taken up computer as their field of study or profession in the last 20 years.

`Exercise:` You may want to filter out responses by age (or age group), if you'd like to analyze and compare the results of the survey for different age groups. Create a new column called `AgeGroup` which contains values like `Less than 10 years`, `10-18 years`, `18-30 years`, `30-45 years`, `45-60 years`, `Older than 60 years`, and repeat the analysis in the rest of this notebook for each group.

In [ ]:
# Define age and bins
bins = [0, 10, 20, 30, 40, 50, 60, 80]
labels = ['less than 10 years', '11-20 years', '21-30 years', '31-40 years', '41-50 years', '51-60 years', 'Older than 60 years']

survey_df['AgeGroup'] = pd.cut(survey_df['Age'], bins=bins, labels=labels, right=False)
# pd.cut() -> pandas function that splits continuous data into bins/categories
# right=False	Interval is [low, high) not including high,	[0-18) means 18 excluded

survey_df['AgeGroup'].value_counts()

In [ ]:
age_counts = survey_df.AgeGroup.value_counts().sort_values()
age_counts

In [ ]:


plt.figure(figsize=(12,6))
plt.title('Age Group Distribution')
plt.xlabel('Age Group')
plt.ylabel('Number of Respondents')
plt.xticks(rotation=45, ha='right')

sns.barplot(x=age_counts.index, y=age_counts.values);

# Gender
Let's look at the distribution of responses for the Gender. It's a well known fact that women and non-binary genders are underrepresented in the programming community, so we might expect to see a skewed distribution here.

In [ ]:
schema.Gender

In [ ]:
gender_counts = survey_df.Gender.value_counts(dropna=False)
gender_counts

A pie chart would be a good way to visualize the distribution.

In [ ]:
gender_counts = survey_df.Gender.value_counts(dropna=False) # dropna=False -> includes NaN values
gender_counts

In [ ]:
plt.figure(figsize=(12,6))
plt.title(schema.Gender)
plt.pie(gender_counts, labels=gender_counts.index, startangle=90)
sns.set_style('darkgrid');

Only about 8% of survey respondents who habe answered the queation identify as women or non-binary. This number is lower than the overall percentage of women and non-binary genders in the programming community - which is estimated to be around 12%.

# Education Level
Formal education in computer science is often considered an important requirement of becoming a programmer. Let't see if this indeed the case, especially since there are many free resources and tutorials available online to learn programming. We'll use a horizontal bar plot to compare education levels of respondents.

In [ ]:
schema.EdLevel

In [ ]:
survey_df.EdLevel.unique()

In [ ]:
survey_df.EdLevel.value_counts()

In [ ]:
plt.figure(figsize=(12,8))
sns.countplot(y=survey_df.EdLevel)
plt.xticks(rotation=75)
plt.title(schema['EdLevel'])
plt.ylabel(None);

In [ ]:
plt.figure(figsize=(12,8))
sns.countplot(y=survey_df.EdLevel)
plt.xticks(rotation=75);
plt.title(schema['EdLevel'])
plt.ylabel(None);

It appears that well over half of the repondents hold a bachelor's or master's degree, so most programmers definitely seem to have some college education, although it's not clear from this graph alone if they hold a degree in computer science.  

**Exercise**: The graph currently shows the number of respondents for each option, can you modify it to show the percentage instead? Further, can you break down the graph to compare the percentages for each degree for men vs women. 

Let's also plot undergraduate majors, but this time we'll convert the number into percentages, and sort by percentage values to make it easier to visualize the order.

In [ ]:
schema.UndergradMajor

In [ ]:
undergrad_pct = survey_df.UndergradMajor.value_counts() * 100 / survey_df.UndergradMajor.count()

In [ ]:
undergrad_pct

In [ ]:
undergrad_pct = survey_df.UndergradMajor.value_counts() *100 / survey_df.UndergradMajor.count()

plt.figure(figsize=(12,8))
sns.barplot(x=undergrad_pct, y=undergrad_pct.index)

plt.title(schema.UndergradMajor)
plt.ylabel(None)
plt.xlabel('Percentage');

It turns that 40% of programmers holding a college degree have a field of study other than computer science - which is very encouraging. This seems to suggest that while college education is helpful in general, you do not need to pursue a major in computer science to become a successful programmer.  

**Exercises:** Analyze the results of the `NEWEdImpt` column for respondents who hold some college degree vs those who don't. Do you notice any difference in opinion?

In [ ]:
schema.NEWEdImpt

In [ ]:
newImpt_pct = survey_df.NEWEdImpt.value_counts() * 100 / survey_df.NEWEdImpt.count()

plt.figure(figsize=(12,6))
sns.barplot(y=newImpt_pct.index, x=newImpt_pct)

plt.title(schema.NEWEdImpt)
plt.ylabel(None)
plt.xlabel('Percentage');

# Employment
Freelancing or contract work is a common choice among programmer, so it would be interesting to compare the breakdowm between full time, part time and freelance work. Let's visualize the data from `Employment` column.

In [ ]:
schema.Employment

In [ ]:
survey_df.Employment.value_counts()

In [ ]:
(survey_df.Employment.value_counts(normalize=True, ascending=True)*100).plot(kind='barh', color='g')
plt.title(schema.Employment)
plt.xlabel('Percentage');

It appears that close to 10% of respondents are employed part time or as freelancers.  

**Exercise:** Add a new column `EmplymentType` which contains values `Enthusiast` (student or not employed but for work), `Professional` (employment full-time, part-time or freelancing) and `Other` (not employed or retired). For each of the graphs that follow, show a comparison between `Enthusiast` and `Professional`.

The `DevType` field contains information about the roles held by respondents. Since the question allows multiple answers, the column contaion lists of values separeted by `;`, which makes it a bit harder to analyze directly.

In [ ]:
schema.DevType

In [ ]:
survey_df.DevType.value_counts()

Let's define a helper function which turns a column containing lists of values (like `survey_df.DevType`) into a data frame with one column for each possible option.

In [ ]:
col_series = survey_df.DevType

In [ ]:
# to_frame() converts a series to a data frame
result_df = col_series.to_frame()
result_df.columns

In [ ]:
def split_multicolumn(col_series):
    result_df = col_series.to_frame()
    options = []
    # Iterate over the column
    for idx, value in col_series[col_series.notnull()].items():
        # Break each value into list of options
        for option in value.split(';'):
            # Add the option as a column to result
            if not option in result_df.columns:
                options.append(option)
                result_df[option] = False
            # Mark the value in the option column as True
            result_df.at[idx, option] = True
    return result_df[options]

In [ ]:
dev_type_df = split_multicolumn(survey_df.DevType)

In [ ]:
survey_df.DevType.nunique()

In [ ]:
dev_type_df

The `dev_type_df` has one column for each option that can be selected as a response. If a responded has selecete the option, the value in the column is `True`, otherwise it is false.  

We can now use the column-wise total to identify the most common roles.

In [ ]:
dev_type_totals = dev_type_df.sum().sort_values(ascending=False)
dev_type_totals

In [ ]:
plt.figure(figsize=(12,8))
sns.barplot(y=dev_type_totals.index, x=dev_type_totals)

plt.ylabel(None)
plt.xlabel('Count');

As one might expect, the most common roles include "Developer" in the name.  

**Exercises:**
- Can you figure out what percentage of respondents work in roles related to data science?
- Which role has the highest percentage of women? 
- Draw conclusions

We've only explored a handful of columns from the 20 columns that we selected. Explore and visualize the remaining columns.

# Asking and Answering Questions
We've already gained several insights about the respondents and the programming community in general, simply by exploring individual column of the dataset. Let's ask some specific questions, and try to answer them using data frame operations and interesting visualizations.

**Q: Which were the most popular programming languages in 2020?**  
To answer this we can use the `LanguageWorkedWith` column. Similar to `DevType` respondents were allowed to choose options here.

In [ ]:
schema.LanguageWorkedWith

In [ ]:
survey_df.LanguageWorkedWith

First, we'll split this column into a data frame containing a column of each languages listed in the options.

In [ ]:
languages_worked_df = split_multicolumn(survey_df.LanguageWorkedWith)

In [ ]:
languages_worked_df

In [ ]:
languages_worked_df.loc[64457]

It appears that a total of 25 languages were included among the options. Let's aggregate these to identify the percentage of respondents who selected each language.

In [ ]:
languages_worked_percentages = languages_worked_df.mean().sort_values(ascending=False) * 100
languages_worked_percentages

We can plot this information using a horizontal bar chart.

In [ ]:
schema.LanguageWorkedWith

In [ ]:
plt.figure(figsize=(12,10))
sns.barplot(x=languages_worked_percentages, y=languages_worked_percentages.index)
plt.title("Languages used in the past year")
plt.ylabel(None)
plt.xlabel("Count");

Perhaps not surprisingly, Javascript & HTML/CSS comes out at the top as web development is one of the most sought skills today and it also happens to be one of the easiest to get started with. SQL is necessary for working with relational databases, so it's no surprise that most programmers work with SQL on a regualar basis. For other forms of development, Python seems to be the popular choice, beating out Java, which was the industry standard for server & application development  for ove 2 decades.

**Exercises:**
- What are the most common languages used by students? How does the list compare with the most common languages used by professional developers?
- What are the most common languages among respondents who do not describe themselves as "Developer, front-end"
- What are the most common languages among respondents who work in fields related to data science?
- What are the most common languages used by developers older than 35 years of age?
- What are the most common languages used by developers in your home country?

**Q: Which languages are the most peolple interested to learn over the next year?**  

For this we can use the `LanguageDesireNextYear` column, with similar processing as the previous one.

In [ ]:
schema.LanguageDesireNextYear

In [ ]:
languages_interested_df = split_multicolumn(survey_df.LanguageDesireNextYear)
languages_interested_percentages = languages_interested_df.mean().sort_values(ascending=False) * 100
languages_interested_percentages

In [ ]:
plt.figure(figsize=(12,12))
sns.barplot(y=languages_interested_percentages.index, x=languages_interested_percentages)
plt.title('Languages people are interested in learning over the next year')
plt.xlabel('Count')
plt.ylabel(None);

Once again, it's not surprising that Python is the language most people are interested in learning - since it is an easy-to-learn general purpose programming language well suited for a variety of domains: application development, numerical computing, data analysis, machine learning, big data, cloud automation, web scraping, scripting etc. We're using Python for this very analysis, so we're in good company.  

**Exercise:** Repeat all the exercises for the previous question, replacing "most common languages" with "languages people are interested in learning/using".

**Q: Which are the most loved languages i.e. a high percentage of people who have used the language want to continue learning and using it over the next year?**  

While this question may seem trick at first, it's really easy to solve using Pandas array operations. Here's what we can do:
- Creat a new data frame `languages_loved_df` which contains a `True` value for a language only is the corresponding values in `languages_worked_df` and `languages_interested_df` are both `True`.
- Take the column wise sum of `languages_loved_df` and divide it by the column-wise sum of `languages_worked_df` to get the percentage of respondents who "love" the language
- Sort the results in decreasing order and plot a horizontal bar graph.

In [ ]:
languages_loved_df = languages_worked_df & languages_interested_df

In [ ]:
languages_loved_df

In [ ]:
languages_loved_percentages = (languages_loved_df.sum() * 100/ languages_worked_df.sum()).sort_values(ascending=False)
languages_loved_percentages

In [ ]:
plt.figure(figsize=(12,10))
sns.barplot(x=languages_loved_percentages, y=languages_loved_percentages.index)
plt.title("Most loved languages")
plt.xlabel("Count");

Rust has been StackOverflow's most-loved language for 4 year in a row, followe by TypeScript which has gained a lot of popularity in the past few years as a good alternative to JavaScript for web development.  

Python features at number 3, despite already being one of the most widely-used languages in the world. This is testament to the fact that the language has solid foundation, is really easy to learn and use, has a strong ecosystem of libraries for various and massive worldwide community of developers to enjoy using it.  

**Exercises:** What are the most dreaded languages i.e. languages which people have used in the past year, but do not want to learn/use over the next year. Hint: `~languages_interested_df`.

**Q: In which countries do developers work the highest number of hours per week? Consider countries with more than 250 responses only.**  

To answer this question, we'll need to use the `groupby` data frame method to aggregate the rows for each country. We'll also need to filter the resulte to only include the countries which have more than 250 respondents.

In [ ]:
countries_df = survey_df.groupby('Country')[['WorkWeekHrs']].mean().sort_values('WorkWeekHrs', ascending=False)

In [ ]:
countries_df

In [ ]:
high_response_countries_df = countries_df.loc[survey_df.Country.value_counts() > 250].head(15)

In [ ]:
high_response_countries_df

The Asian countries like Iran, China & Israel have the highest working hours, followed by the United States. However, there isn't too much variation overall and the average working hours seem to be around 40 hours per week.  

**Exercises:**  

- How does average work hours compare across continents? 
- Which role has the highest average number of hours worked per week? Which role has the lowest?
- How do the hours worked compare between freelancers and developers working full-time?

**Q: How important is it to start young to build a career in programming?**  

Let's create a scatter plot of `Age` vs `YearsCodePro` (i.e. years coding experience) to answer this question.

In [ ]:
schema.YearsCodePro

In [ ]:
sns.scatterplot(x='Age', y='YearsCodePro', hue='Hobbyist', data=survey_df)
plt.xlabel("Age")
plt.ylabel("Years of professional coding experience");

You can see points all over the graph, which seems to indicate that you can **start programming professionally at any age**. Also, many people who have been coding for several decades professionally also seem to enjoy it as a hobby.  

We can also view the distribution of `Age1stCode` column to see when the respondents tried programming for the first time.

In [ ]:
plt.title(schema.Age1stCode)
sns.distplot(survey_df.Age1stCode);

As you might expect, most people seem to have had some exposure to programming before the age of 40, but there are people of all ages and walks of life who are learning to code.  

**Exercises:** 

- How does experience change opinions & preferences? Repeat the entire analysis while comparing the responses of people who have more than 10 years of professional programming experience vs those who don't. Do you see any interesting trends?
- Compare the years of professional coding experience across different genders.

We've barely scratched the surface here, and hopefully you are already thinking of many more questions that you'd like to answer using this data

# Inferences and Conclusions
We've drawn many interesting inferences from the survey, here's a summary of the few of them:  

- Based on the demographics of the survey respondents, we can infer that the survey is somewhat representative of the overall programming community, although it definitely has fewer reponses from programmers in non-English-speaking countries and from women & non-binary genders.

- The programming community is not as diverse as it can be, and although things are improving, we should take more efforts to support & encourage members of underrepresented communities - whether it is in terms of age, country, race, gender or otherwise.

- Most programmers hold a college degree, although a fairly large percentage did not have computer science as their major in college, so a computer science degree isn't compulsory for learning to code or building a career in programming.

- A significant percentage of programmers either work part time or as freelancers, and this can be a great way to break into the field, especially when you're just getting started.

- Javascript &  HTML/CSS are the most used programming languages in 2020, closely followed by SQL & Python.

- Python is the language most people are intereste in learnig - since it is an easy-to-learn general purpose programming language well suited for a variety of domains.

- Rust and TypeScript are the most "loved" languages in 2020, both of which have small but fast-growing communities. Python is a close third, despite already being a widely used language.

- Programmers around the world seems to be working for around 40 hours a week on average, with slight variations by country.

- You can learn and start programming professionally at any age, and you're likely to have a long and fulfilling career if you also enjoy programming as a hobby.